# Проект 3 — Уровень дохода (Adult Census)
Классификация дохода `>50K` по данным Adult Census. Выполняются базовый, основной и продвинутый блоки задания.

**Важно:** если `adult.csv` отсутствует, запустите `download_data.py` или выполните первую ячейку — данные будут загружены из UCI.

In [ ]:
import warnings
warnings.filterwarnings('ignore')
import numpy as np, pandas as pd, matplotlib.pyplot as plt, seaborn as sns
from pathlib import Path
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_validate, cross_val_predict
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import (accuracy_score, balanced_accuracy_score, precision_score, recall_score,
    f1_score, roc_auc_score, confusion_matrix, classification_report, ConfusionMatrixDisplay)
from imblearn.pipeline import Pipeline as ImbPipeline
from imblearn.over_sampling import RandomOverSampler

RANDOM_STATE = 42
DATA_PATH = Path('adult.csv')
if not DATA_PATH.exists():
    import urllib.request
    url = 'https://archive.ics.uci.edu/ml/machine-learning-databases/adult/adult.data'
    columns = ['age','workclass','fnlwgt','education','education.num','marital.status','occupation',
      'relationship','race','sex','capital.gain','capital.loss','hours.per.week','native.country','income']
    raw = pd.read_csv(url, header=None, names=columns, na_values='?', skipinitialspace=True)
    raw.to_csv(DATA_PATH, index=False)
    print('Датасет загружен из UCI:', DATA_PATH)

df = pd.read_csv(DATA_PATH, na_values='?', skipinitialspace=True)
df.columns = df.columns.str.strip()
df['income'] = df['income'].astype(str).str.strip().str.replace('.', '', regex=False)
df['income'] = (df['income'] == '>50K').astype(int)
print('Размер:', df.shape)
display(df.head())
df.info()


## 1. Базовый блок: первичный анализ и baseline

In [ ]:
missing = pd.DataFrame({'missing_count':df.isna().sum(),'missing_percent':(df.isna().mean()*100).round(2)})
display(missing.sort_values('missing_count',ascending=False))
print('Дубликаты:', df.duplicated().sum())
display(df['income'].value_counts().rename(index={0:'<=50K',1:'>50K'}).to_frame('count'))
print('Доли классов:')
display((df['income'].value_counts(normalize=True)*100).round(2))
X, y = df.drop(columns='income'), df['income']
X_train, X_test, y_train, y_test = train_test_split(X,y,test_size=.2,stratify=y,random_state=RANDOM_STATE)
# education — текстовый дубль education.num, исключаем его до обучения.
X_train = X_train.drop(columns=['education'], errors='ignore')
X_test = X_test.drop(columns=['education'], errors='ignore')
cat_cols = X_train.select_dtypes(include=['object','category']).columns.tolist()
num_cols = X_train.select_dtypes(include=np.number).columns.tolist()
num_pipe = Pipeline([('imputer',SimpleImputer(strategy='median')),('scaler',StandardScaler())])
cat_pipe = Pipeline([('imputer',SimpleImputer(strategy='most_frequent')),('onehot',OneHotEncoder(handle_unknown='ignore'))])
preprocessor = ColumnTransformer([('num',num_pipe,num_cols),('cat',cat_pipe,cat_cols)])
baseline = Pipeline([('prep',preprocessor),('model',LogisticRegression(max_iter=2000,random_state=RANDOM_STATE))])
baseline.fit(X_train,y_train)
pred = baseline.predict(X_test)
print('Baseline accuracy:',round(accuracy_score(y_test,pred),4))
print(classification_report(y_test,pred,target_names=['<=50K','>50K']))
ConfusionMatrixDisplay.from_predictions(y_test,pred,display_labels=['<=50K','>50K'])
plt.show()


## 2. EDA и подготовка данных

In [ ]:
display(X_train[num_cols].describe().T)
for col in ['age','hours.per.week','education.num']:
    if col in X_train:
        sns.histplot(data=pd.DataFrame({'value':X_train[col],'income':y_train}),x='value',hue='income',bins=30,stat='density',common_norm=False)
        plt.title(col); plt.show()
for col in ['workclass','marital.status','occupation']:
    if col in X_train:
        display(pd.crosstab(X_train[col].fillna('Missing'),y_train,normalize='index').round(3))


## 3. Основной блок: CV, две модели и дисбаланс классов
Все преобразования и oversampling находятся внутри Pipeline, поэтому обучаются только на соответствующем train-fold.

In [ ]:
cv = StratifiedKFold(n_splits=5,shuffle=True,random_state=RANDOM_STATE)
scoring = {'accuracy':'accuracy','balanced_accuracy':'balanced_accuracy','f1':'f1','roc_auc':'roc_auc'}
def make_model(kind,balance):
    if kind=='Logistic Regression':
        clf=LogisticRegression(max_iter=2000,class_weight='balanced' if balance=='class_weight' else None,random_state=RANDOM_STATE)
    else:
        clf=DecisionTreeClassifier(max_depth=12,min_samples_leaf=10,class_weight='balanced' if balance=='class_weight' else None,random_state=RANDOM_STATE)
    steps=[('prep',preprocessor)]
    if balance=='oversampling':
        return ImbPipeline(steps+[('sampler',RandomOverSampler(random_state=RANDOM_STATE)),('model',clf)])
    return Pipeline(steps+[('model',clf)])
rows=[]
for kind in ['Logistic Regression','Decision Tree']:
    for balance in ['none','class_weight','oversampling']:
        scores=cross_validate(make_model(kind,balance),X_train,y_train,cv=cv,scoring=scoring,n_jobs=1)
        rows.append({'model':kind,'imbalance_method':balance,**{m:scores['test_'+m].mean() for m in scoring}})
cv_results=pd.DataFrame(rows).sort_values('accuracy',ascending=False)
display(cv_results.round(4))


## 4. Подбор порога и итоговая оценка

In [ ]:
best=cv_results.iloc[0]
selected=make_model(best['model'],best['imbalance_method'])
oof=cross_val_predict(selected,X_train,y_train,cv=cv,method='predict_proba',n_jobs=1)[:,1]
thresholds=np.linspace(.2,.8,121)
# Порог выбирается только на out-of-fold прогнозах обучающей выборки.
threshold_scores=[accuracy_score(y_train,(oof>=t).astype(int)) for t in thresholds]
best_threshold=float(thresholds[int(np.argmax(threshold_scores))])
selected.fit(X_train,y_train)
proba=selected.predict_proba(X_test)[:,1]

def metrics(y_true,p,label):
    return {'variant':label,'accuracy':accuracy_score(y_true,p),'balanced_accuracy':balanced_accuracy_score(y_true,p),
      'precision_>50K':precision_score(y_true,p,zero_division=0),'recall_>50K':recall_score(y_true,p,zero_division=0),
      'f1_>50K':f1_score(y_true,p,zero_division=0),'roc_auc':roc_auc_score(y_true,proba)}
pred05=(proba>=.5).astype(int); pred_t=(proba>=best_threshold).astype(int)
display(pd.DataFrame([metrics(y_test,pred05,'threshold 0.50'),metrics(y_test,pred_t,f'threshold {best_threshold:.3f}')]).round(4))
print('Selected:',best['model'],'| imbalance:',best['imbalance_method'],'| threshold:',round(best_threshold,3))
print(classification_report(y_test,pred_t,target_names=['<=50K','>50K']))
ConfusionMatrixDisplay.from_predictions(y_test,pred_t,display_labels=['<=50K','>50K']); plt.show()


## 5. Продвинутый блок: анализ ошибок и ограничения

In [ ]:
analysis=X_test.copy(); analysis['actual']=y_test.to_numpy(); analysis['predicted']=pred_t
analysis['error']=(analysis.actual!=analysis.predicted).astype(int)
analysis['age_group']=pd.cut(analysis.age,[0,25,35,45,55,65,100],labels=['<=25','26-35','36-45','46-55','56-65','66+'])
for group_col in ['sex','race','age_group']:
    rows=[]
    for group,part in analysis.groupby(group_col,observed=False):
        neg=part[part.actual==0]; pos=part[part.actual==1]
        rows.append({'group':str(group),'n':len(part),'error_rate':part.error.mean(),
          'FPR':(neg.predicted==1).mean() if len(neg) else np.nan,
          'FNR':(pos.predicted==0).mean() if len(pos) else np.nan})
    print('Ошибки по группам:',group_col)
    display(pd.DataFrame(rows).round(3))


## Выводы
Сравните модели по метрике заказчика и стоимости двух типов ошибок. Для премиальной кредитной карты отдельно оцените false positive (предложение человеку с низким доходом) и false negative (упущенная возможность). Необходимо проверить стабильность метрик, репрезентативность данных, калибровку вероятностей и правовые ограничения. Пол, раса и другие чувствительные признаки нельзя автоматически использовать для реального кредитного решения. Это учебная модель, а не рекомендация банку к внедрению.